In [1]:
# 1. Force update and install Java 8
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# 2. Re-install the correct library versions
!pip install -q "pyspark[connect]==4.0.0" pydeequ dataproc-spark-connect

# # 3. Set Environment Variables dynamically
# import os

# # This command finds the actual path of the Java you just installed
# java_path = !readlink -f /usr/bin/java | sed "s:bin/java::"
# os.environ["JAVA_HOME"] = java_path[0]

# # Set this for PyDeequ compatibility
# os.environ["SPARK_VERSION"] = "3.5"

# print(f"✅ JAVA_HOME set to: {os.environ['JAVA_HOME']}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
import sys

# 3. Ask the system for the REAL Java 8 path
# This replaces the hardcoded Step 3 that was causing the crash
java_8_cmd = !readlink -f /usr/bin/java | sed "s:bin/java::"
os.environ["JAVA_HOME"] = java_8_cmd[0]

# 4. Set necessary environment variables
os.environ["SPARK_VERSION"] = "3.5"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pydeequ
from pyspark.sql import SparkSession

# 5. Robust Spark Initialization
# Pinning the specific Deequ JAR for Spark 4.0.0/3.5 compatibility
DEEQU_MAVEN_COORD = "com.amazon.deequ:deequ:2.0.4-spark-3.5"

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1") # Crucial for Colab stability
    .config("spark.jars.packages", DEEQU_MAVEN_COORD)
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate())

print(f"✅ JAVA_HOME identified as: {os.environ['JAVA_HOME']}")
print(f"✅ Spark version {spark.version} is live!")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [2]:
# # 4. Initialize Spark Session with Explicit Deequ JARs
# import pydeequ
# from pyspark.sql import SparkSession

# # Use a stable Maven coordinate for Deequ
# # Note: '2.0.4-spark-3.5' is currently the most stable for Spark 3.5/4.x environments
# DEEQU_MAVEN_COORD = "com.amazon.deequ:deequ:2.0.4-spark-3.5"

# spark = (SparkSession.builder
#     .master("local[*]")
#     .config("spark.driver.host", "127.0.0.1")
#     .config("spark.jars.packages", DEEQU_MAVEN_COORD) # Force the specific JAR
#     .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
#     .config("spark.sql.execution.arrow.pyspark.enabled", "true")
#     .getOrCreate())

# # Verify the JVM actually has the classes loaded
# try:
#     spark._jvm.com.amazon.deequ.checks.Check
#     print("✅ Deequ Java classes successfully loaded in JVM!")
# except:
#     print("❌ Deequ classes still missing. Try restarting the runtime.")

# # print(f"✅ Spark version: {spark.version}")

# # Verify everything is hooked up correctly
# print(f"✅ Spark version: {spark.version}")
# print(f"✅ PyDeequ mapped to: {os.environ.get('SPARK_VERSION')}")
# print("🚀 Spark Session with PyDeequ is ready!")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
# Latest 3 months available as of March 2026
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-09.parquet" -O "sep_2025.parquet"
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-10.parquet" -O "oct_2025.parquet"
!wget -q "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet" -O "nov_2025.parquet"
!wget -q "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv" -O "zone_lookup.csv"

print("✅ All files downloaded")
!ls -lh *.parquet *.csv

In [ ]:
# Load all three months
df_sep = spark.read.parquet("sep_2025.parquet")
df_oct = spark.read.parquet("oct_2025.parquet")
df_nov = spark.read.parquet("nov_2025.parquet")

# Combine into one DataFrame
df = df_sep.union(df_oct).union(df_nov)

print(f"✅ Sep rows: {df_sep.count():,}")
print(f"✅ Oct rows: {df_oct.count():,}")
print(f"✅ Nov rows: {df_nov.count():,}")
print(f"✅ Total combined rows: {df.count():,}")
print(f"✅ Columns: {len(df.columns)}")
print()
df.printSchema()
df.show(5, truncate=False)

In [ ]:
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite, VerificationResult

check = Check(spark, CheckLevel.Warning, "NYC Taxi Quality Checks")

checkResult = (VerificationSuite(spark)
    .onData(df)
    .addCheck(
        check
        # Completeness checks
        .isComplete("tpep_pickup_datetime")
        .isComplete("tpep_dropoff_datetime")
        .isComplete("fare_amount")
        # Value validity checks
        .isNonNegative("fare_amount")
        .isNonNegative("tip_amount")
        .isNonNegative("trip_distance")
        # Business rule checks
        .satisfies("trip_distance > 0", "Trip distance must be positive")
        .satisfies("fare_amount > 0", "Revenue integrity: fare must be positive")
        .satisfies("tpep_dropoff_datetime > tpep_pickup_datetime", "Dropoff must be after pickup")
        .satisfies("passenger_count >= 1 AND passenger_count <= 6", "Passenger count must be 1-6")
        .satisfies("PULocationID >= 1 AND PULocationID <= 265", "Pickup zone must be valid NYC zone")
        .satisfies("DOLocationID >= 1 AND DOLocationID <= 265", "Dropoff zone must be valid NYC zone")
    ).run()
)

print("✅ Checks complete")

In [ ]:
results_df = VerificationResult.checkResultsAsDataFrame(spark, checkResult)
results_df.select("constraint", "constraint_status", "constraint_message").show(20, truncate=False)

In [ ]:
import pandas as pd

pdf = results_df.toPandas()
total = len(pdf)
passed = len(pdf[pdf["constraint_status"] == "Success"])
failed = total - passed

print("=" * 55)
print("  NYC Taxi Data Quality Report — Sep/Oct/Nov 2025")
print("=" * 55)
print(f"  Total checks : {total}")
print(f"  ✅ Passed    : {passed}")
print(f"  🚨 Failed    : {failed}")
print("=" * 55)
if failed > 0:
    print("\n  Failed checks:")
    for _, row in pdf[pdf["constraint_status"] != "Success"].iterrows():
        print(f"  → {row['constraint']}")

In [ ]:
from pyspark.sql.functions import col, count, when

print("📊 Quantifying data quality issues...\n")

total_rows = df.count()

issues = {
    "Negative fare_amount"        : df.filter(col("fare_amount") < 0).count(),
    "Zero or negative fare"       : df.filter(col("fare_amount") <= 0).count(),
    "Negative tip_amount"         : df.filter(col("tip_amount") < 0).count(),
    "Zero trip distance"          : df.filter(col("trip_distance") <= 0).count(),
    "Invalid passenger count"     : df.filter((col("passenger_count") < 1) | (col("passenger_count") > 6)).count(),
    "Dropoff before pickup"       : df.filter(col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime")).count(),
}

print(f"Total rows analyzed: {total_rows:,}\n")
print(f"{'Issue':<35} {'Rows Affected':>15} {'% of Data':>10}")
print("-" * 63)
for issue, count_val in issues.items():
    pct = (count_val / total_rows) * 100
    print(f"{issue:<35} {count_val:>15,} {pct:>9.2f}%")